# Notebook 09 — Deep Learning Models

This notebook builds neural-network baselines for S&P 500 next-day return-direction prediction.

**Input**
`data/raw/sp500_1950_present.csv`

**Models**
- Multilayer Perceptron (MLP)
- LSTM
- GRU

**Target**
- Next-day return direction: `1` when the next trading day's close-to-close return is positive, otherwise `0`

**Methodology**
- Chronological train/test split
- Sequence construction preserves temporal order
- Scalers are fitted only on the training period
- No random shuffling across train/test
- Early stopping is used on a chronological validation segment
- Test data remains untouched until final evaluation

The notebook is designed to work on a normal CPU-capable Windows development machine. It does not require a GPU.


## 1. Imports

In [ ]:
from pathlib import Path
import json
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings("ignore")

print("Imports loaded successfully.")
print("PyTorch version:", torch.__version__)


## 2. Reproducibility and Device

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Seed:", SEED)
print("Device:", DEVICE)


## 3. Configuration and Paths

In [ ]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "data").exists():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/mnt/data/quant-trading-research"),
    ]
    for candidate in candidates:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            PROJECT_ROOT = candidate
            break

MASTER_PATH = PROJECT_ROOT / "data" / "raw" / "sp500_1950_present.csv"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
TABLE_DIR = PROJECT_ROOT / "reports" / "tables"
REPORT_DIR = PROJECT_ROOT / "reports" / "generated"
MODEL_DIR = PROJECT_ROOT / "models" / "deep_learning"

for path in [
    INTERIM_DIR,
    FIGURE_DIR,
    TABLE_DIR,
    REPORT_DIR,
    MODEL_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

EXPECTED_COLUMNS = [
    "Date", "Open", "High", "Low", "Close", "Adj.Close", "Volume"
]

TEST_FRACTION = 0.20
VALIDATION_FRACTION_WITHIN_TRAIN = 0.15
SEQUENCE_LENGTH = 30

BATCH_SIZE = 128
EPOCHS = 30
PATIENCE = 5
LEARNING_RATE = 1e-3

print(f"Master dataset: {MASTER_PATH}")
print(f"Sequence length: {SEQUENCE_LENGTH}")


## 4. Load and Validate the Master Dataset

In [ ]:
if not MASTER_PATH.exists():
    raise FileNotFoundError(
        f"Master dataset not found: {MASTER_PATH}. "
        "Run the earlier notebooks first."
    )

df = pd.read_csv(MASTER_PATH, low_memory=False)

if list(df.columns) != EXPECTED_COLUMNS:
    raise ValueError(
        f"Unexpected schema. Expected {EXPECTED_COLUMNS}; "
        f"received {list(df.columns)}"
    )

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

for column in EXPECTED_COLUMNS[1:]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df = df.sort_values("Date").reset_index(drop=True)

if df["Date"].isna().any():
    raise ValueError("Invalid dates detected.")

if df["Date"].duplicated().any():
    raise ValueError("Duplicate dates detected.")

if df[["Open", "High", "Low", "Close", "Adj.Close", "Volume"]].isna().any().any():
    raise ValueError("Missing OHLCV values detected.")

print(f"Rows: {len(df):,}")
print(f"Date range: {df['Date'].min().date()} → {df['Date'].max().date()}")


## 5. Build Backward-Looking Features

The neural networks receive normalized market features rather than raw price levels.

All features use only information available at the current observation.


In [ ]:
df["return_1d"] = df["Close"].pct_change()
df["return_5d"] = df["Close"].pct_change(5)
df["return_21d"] = df["Close"].pct_change(21)
df["return_63d"] = df["Close"].pct_change(63)

df["volatility_5d"] = (
    df["return_1d"].rolling(5).std() * np.sqrt(252)
)

df["volatility_21d"] = (
    df["return_1d"].rolling(21).std() * np.sqrt(252)
)

df["volatility_63d"] = (
    df["return_1d"].rolling(63).std() * np.sqrt(252)
)

df["sma_20"] = df["Close"].rolling(20).mean()
df["sma_50"] = df["Close"].rolling(50).mean()
df["sma_200"] = df["Close"].rolling(200).mean()

df["price_to_sma_20"] = (
    df["Close"] / df["sma_20"] - 1
)

df["price_to_sma_50"] = (
    df["Close"] / df["sma_50"] - 1
)

df["price_to_sma_200"] = (
    df["Close"] / df["sma_200"] - 1
)

df["sma_50_vs_sma_200"] = (
    df["sma_50"] / df["sma_200"] - 1
)

df["range_pct"] = (
    (df["High"] - df["Low"]) / df["Close"]
)

df["intraday_return"] = (
    df["Close"] / df["Open"] - 1
)

df["overnight_return"] = (
    df["Open"] / df["Close"].shift(1) - 1
)

df["volume_ratio_20"] = (
    df["Volume"] /
    df["Volume"].rolling(20).mean()
)

df["volume_ratio_63"] = (
    df["Volume"] /
    df["Volume"].rolling(63).mean()
)

feature_columns = [
    "return_1d",
    "return_5d",
    "return_21d",
    "return_63d",
    "volatility_5d",
    "volatility_21d",
    "volatility_63d",
    "price_to_sma_20",
    "price_to_sma_50",
    "price_to_sma_200",
    "sma_50_vs_sma_200",
    "range_pct",
    "intraday_return",
    "overnight_return",
    "volume_ratio_20",
    "volume_ratio_63",
]

print(f"Feature count: {len(feature_columns)}")


## 6. Create the Next-Day Direction Target

In [ ]:
df["next_day_return"] = (
    df["Close"].shift(-1) / df["Close"] - 1
)

df["target"] = (
    df["next_day_return"] > 0
).astype(int)

model_df = df[
    ["Date", "Close", "next_day_return", "target"] + feature_columns
].dropna(
    subset=feature_columns + ["next_day_return"]
).reset_index(drop=True)

print(f"Modeling observations: {len(model_df):,}")
print(
    "Target distribution:",
    model_df["target"].value_counts(normalize=True)
    .sort_index()
    .to_dict()
)


## 7. Chronological Train/Test Split

In [ ]:
split_idx = int(
    len(model_df) * (1 - TEST_FRACTION)
)

train_df = model_df.iloc[:split_idx].copy()
test_df = model_df.iloc[split_idx:].copy()

print(f"Train rows: {len(train_df):,}")
print(f"Test rows: {len(test_df):,}")
print(f"Train end: {train_df['Date'].max().date()}")
print(f"Test start: {test_df['Date'].min().date()}")
print(f"Test end: {test_df['Date'].max().date()}")


## 8. Fit the Feature Scaler on Training Data Only

In [ ]:
feature_scaler = StandardScaler()

feature_scaler.fit(
    train_df[feature_columns]
)

scaled_all_features = feature_scaler.transform(
    model_df[feature_columns]
)

scaled_all_features = pd.DataFrame(
    scaled_all_features,
    columns=feature_columns,
)

print("Scaler fitted using training-period observations only.")


## 9. Create Chronological Sequences

Each sequence contains the previous `SEQUENCE_LENGTH` observations.

The sequence ending at time `t` predicts the direction of the return from `t` to `t+1`.

Sequence construction does not shuffle observations.


In [ ]:
feature_values = scaled_all_features.to_numpy(
    dtype=np.float32
)

target_values = model_df["target"].to_numpy(
    dtype=np.float32
)

date_values = model_df["Date"].to_numpy()

X_sequences = []
y_sequences = []
sequence_dates = []

for i in range(
    SEQUENCE_LENGTH - 1,
    len(model_df)
):
    start = i - SEQUENCE_LENGTH + 1
    end = i + 1

    X_sequences.append(
        feature_values[start:end]
    )

    y_sequences.append(
        target_values[i]
    )

    sequence_dates.append(
        date_values[i]
    )

X_sequences = np.asarray(
    X_sequences,
    dtype=np.float32
)

y_sequences = np.asarray(
    y_sequences,
    dtype=np.float32
)

sequence_dates = pd.to_datetime(
    sequence_dates
)

print("Sequence tensor:", X_sequences.shape)
print("Target tensor:", y_sequences.shape)


## 10. Split Sequences Chronologically

The split is based on the original training/test boundary.

Within the training period, the last 15% is used for validation.

The test period is never used for training or early stopping.


In [ ]:
train_cutoff_date = train_df["Date"].max()

train_mask = sequence_dates <= train_cutoff_date
test_mask = sequence_dates > train_cutoff_date

X_train_full = X_sequences[train_mask]
y_train_full = y_sequences[train_mask]

X_test = X_sequences[test_mask]
y_test = y_sequences[test_mask]

train_sequence_dates = sequence_dates[train_mask]
test_sequence_dates = sequence_dates[test_mask]

validation_split_idx = int(
    len(X_train_full) *
    (1 - VALIDATION_FRACTION_WITHIN_TRAIN)
)

X_train = X_train_full[:validation_split_idx]
y_train = y_train_full[:validation_split_idx]

X_val = X_train_full[validation_split_idx:]
y_val = y_train_full[validation_split_idx:]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)


## 11. Verify Temporal Boundaries

In [ ]:
print(
    "Training end:",
    pd.Timestamp(train_sequence_dates[validation_split_idx - 1]).date()
)

print(
    "Validation start:",
    pd.Timestamp(train_sequence_dates[validation_split_idx]).date()
)

print(
    "Validation end:",
    pd.Timestamp(train_sequence_dates[-1]).date()
)

print(
    "Test start:",
    pd.Timestamp(test_sequence_dates[0]).date()
)

assert (
    pd.Timestamp(train_sequence_dates[-1])
    <
    pd.Timestamp(test_sequence_dates[0])
)

print("Temporal separation check: PASS")


## 12. Convert Sequences to PyTorch DataLoaders

In [ ]:
train_dataset = TensorDataset(
    torch.tensor(X_train),
    torch.tensor(y_train),
)

val_dataset = TensorDataset(
    torch.tensor(X_val),
    torch.tensor(y_val),
)

test_dataset = TensorDataset(
    torch.tensor(X_test),
    torch.tensor(y_test),
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print("DataLoaders created.")


## 13. Define the MLP Model

The MLP receives the complete flattened sequence.

It provides a non-recurrent neural baseline.


In [ ]:
class MLPClassifier(nn.Module):
    def __init__(self, sequence_length, n_features):
        super().__init__()

        input_dim = sequence_length * n_features

        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        x = x.reshape(x.size(0), -1)
        return self.network(x).squeeze(1)


## 14. Define the LSTM Model

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(
        self,
        n_features,
        hidden_size=64,
        num_layers=2,
        dropout=0.20,
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        output, _ = self.lstm(x)
        last_output = output[:, -1, :]
        last_output = self.dropout(last_output)
        return self.fc(last_output).squeeze(1)


## 15. Define the GRU Model

In [ ]:
class GRUClassifier(nn.Module):
    def __init__(
        self,
        n_features,
        hidden_size=64,
        num_layers=2,
        dropout=0.20,
    ):
        super().__init__()

        self.gru = nn.GRU(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        output, _ = self.gru(x)
        last_output = output[:, -1, :]
        last_output = self.dropout(last_output)
        return self.fc(last_output).squeeze(1)


## 16. Class Imbalance Handling

The positive/negative classes are usually close to balanced, but the training loss uses a calculated positive-class weight when needed.

The weight is computed from the training labels only.


In [ ]:
positive_count = float(y_train.sum())
negative_count = float(len(y_train) - y_train.sum())

if positive_count > 0:
    pos_weight_value = negative_count / positive_count
else:
    pos_weight_value = 1.0

pos_weight = torch.tensor(
    [pos_weight_value],
    dtype=torch.float32,
    device=DEVICE,
)

print("Training positive count:", int(positive_count))
print("Training negative count:", int(negative_count))
print("Positive-class weight:", float(pos_weight_value))


## 17. Training Utilities

In [ ]:
def make_model(model_name):
    if model_name == "MLP":
        return MLPClassifier(
            sequence_length=SEQUENCE_LENGTH,
            n_features=len(feature_columns),
        )

    if model_name == "LSTM":
        return LSTMClassifier(
            n_features=len(feature_columns),
        )

    if model_name == "GRU":
        return GRUClassifier(
            n_features=len(feature_columns),
        )

    raise ValueError(f"Unknown model: {model_name}")


def evaluate_epoch(
    model,
    loader,
    criterion,
):
    model.eval()

    total_loss = 0.0
    total_items = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            logits = model(xb)
            loss = criterion(logits, yb)

            batch_size = len(yb)

            total_loss += (
                loss.item() * batch_size
            )
            total_items += batch_size

    return total_loss / max(total_items, 1)


def train_model(
    model_name,
    epochs=EPOCHS,
    patience=PATIENCE,
):
    model = make_model(model_name).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=1e-4,
    )

    history = []
    best_val_loss = np.inf
    best_state = None
    patience_counter = 0

    for epoch in range(1, epochs + 1):
        model.train()

        running_loss = 0.0
        total_items = 0

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            optimizer.step()

            batch_size = len(yb)

            running_loss += (
                loss.item() * batch_size
            )
            total_items += batch_size

        train_loss = (
            running_loss /
            max(total_items, 1)
        )

        val_loss = evaluate_epoch(
            model,
            val_loader,
            criterion,
        )

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
        })

        print(
            f"{model_name} | "
            f"Epoch {epoch:02d}/{epochs} | "
            f"train_loss={train_loss:.5f} | "
            f"val_loss={val_loss:.5f}"
        )

        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(
                f"Early stopping at epoch {epoch}."
            )
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    history_df = pd.DataFrame(history)

    return model, history_df


## 18. Train MLP

In [ ]:
mlp_model, mlp_history = train_model(
    "MLP"
)

print("Best MLP validation loss:",
      mlp_history["val_loss"].min())


## 19. Train LSTM

In [ ]:
lstm_model, lstm_history = train_model(
    "LSTM"
)

print("Best LSTM validation loss:",
      lstm_history["val_loss"].min())


## 20. Train GRU

In [ ]:
gru_model, gru_history = train_model(
    "GRU"
)

print("Best GRU validation loss:",
      gru_history["val_loss"].min())


## 21. Plot Training Curves — MLP

In [ ]:
fig = plt.figure(figsize=(10, 6))

plt.plot(
    mlp_history["epoch"],
    mlp_history["train_loss"],
    label="Train"
)

plt.plot(
    mlp_history["epoch"],
    mlp_history["val_loss"],
    label="Validation"
)

plt.title("MLP Training Curve")
plt.xlabel("Epoch")
plt.ylabel("BCE Loss")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()

path = FIGURE_DIR / "sp500_mlp_training_curve.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {path}")


## 22. Plot Training Curves — LSTM

In [ ]:
fig = plt.figure(figsize=(10, 6))

plt.plot(
    lstm_history["epoch"],
    lstm_history["train_loss"],
    label="Train"
)

plt.plot(
    lstm_history["epoch"],
    lstm_history["val_loss"],
    label="Validation"
)

plt.title("LSTM Training Curve")
plt.xlabel("Epoch")
plt.ylabel("BCE Loss")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()

path = FIGURE_DIR / "sp500_lstm_training_curve.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {path}")


## 23. Plot Training Curves — GRU

In [ ]:
fig = plt.figure(figsize=(10, 6))

plt.plot(
    gru_history["epoch"],
    gru_history["train_loss"],
    label="Train"
)

plt.plot(
    gru_history["epoch"],
    gru_history["val_loss"],
    label="Validation"
)

plt.title("GRU Training Curve")
plt.xlabel("Epoch")
plt.ylabel("BCE Loss")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()

path = FIGURE_DIR / "sp500_gru_training_curve.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {path}")


## 24. Neural-Network Prediction Utility

In [ ]:
def predict_probabilities(model, loader):
    model.eval()

    all_probabilities = []
    all_targets = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)

            logits = model(xb)

            probabilities = torch.sigmoid(
                logits
            ).detach().cpu().numpy()

            all_probabilities.extend(
                probabilities
            )

            all_targets.extend(
                yb.numpy()
            )

    return (
        np.asarray(all_probabilities),
        np.asarray(all_targets),
    )


## 25. Generate Test Predictions

In [ ]:
neural_models = {
    "MLP": mlp_model,
    "LSTM": lstm_model,
    "GRU": gru_model,
}

neural_probabilities = {}
neural_predictions = {}
neural_metrics = []

for name, model in neural_models.items():
    probabilities, targets = predict_probabilities(
        model,
        test_loader,
    )

    predictions = (
        probabilities >= 0.50
    ).astype(int)

    neural_probabilities[name] = probabilities
    neural_predictions[name] = predictions

    neural_metrics.append({
        "model": name,
        "accuracy": accuracy_score(
            targets,
            predictions,
        ),
        "precision": precision_score(
            targets,
            predictions,
            zero_division=0,
        ),
        "recall": recall_score(
            targets,
            predictions,
            zero_division=0,
        ),
        "f1": f1_score(
            targets,
            predictions,
            zero_division=0,
        ),
        "roc_auc": roc_auc_score(
            targets,
            probabilities,
        ),
    })

neural_metrics_df = pd.DataFrame(
    neural_metrics
).sort_values(
    "roc_auc",
    ascending=False
).reset_index(drop=True)

display(neural_metrics_df)


## 26. Classification Reports

In [ ]:
for name in neural_models:
    print("=" * 72)
    print(name)
    print("=" * 72)

    print(
        classification_report(
            y_test,
            neural_predictions[name],
            target_names=["Down/Flat", "Up"],
            zero_division=0,
        )
    )


## 27. Confusion Matrices

In [ ]:
for name in neural_models:
    cm = confusion_matrix(
        y_test,
        neural_predictions[name],
    )

    print(name)

    display(
        pd.DataFrame(
            cm,
            index=["Actual Down/Flat", "Actual Up"],
            columns=[
                "Predicted Down/Flat",
                "Predicted Up",
            ],
        )
    )


## 28. Neural Model Comparison

In [ ]:
neural_metrics_df.to_csv(
    TABLE_DIR / "sp500_deep_learning_model_comparison.csv",
    index=False,
)

display(neural_metrics_df)


## 29. Neural Model ROC-AUC Plot

In [ ]:
fig = plt.figure(figsize=(10, 6))

plt.bar(
    neural_metrics_df["model"],
    neural_metrics_df["roc_auc"],
)

plt.title("Deep Learning Model ROC-AUC")
plt.xlabel("Model")
plt.ylabel("ROC-AUC")
plt.grid(True, axis="y", alpha=0.25)
plt.tight_layout()

path = FIGURE_DIR / "sp500_deep_learning_roc_auc.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {path}")


## 30. Neural Model F1 Plot

In [ ]:
fig = plt.figure(figsize=(10, 6))

plt.bar(
    neural_metrics_df["model"],
    neural_metrics_df["f1"],
)

plt.title("Deep Learning Model F1 Score")
plt.xlabel("Model")
plt.ylabel("F1")
plt.grid(True, axis="y", alpha=0.25)
plt.tight_layout()

path = FIGURE_DIR / "sp500_deep_learning_f1.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {path}")


## 31. Prediction Probability Timeline

In [ ]:
best_dl_model = neural_metrics_df.iloc[0]["model"]

best_dl_probability = neural_probabilities[
    best_dl_model
]

fig = plt.figure(figsize=(14, 6))

plt.plot(
    test_sequence_dates,
    best_dl_probability,
)

plt.axhline(
    0.50,
    linestyle="--",
    label="0.50 threshold",
)

plt.title(
    f"{best_dl_model} — "
    "Probability of Positive Next-Day Return"
)

plt.xlabel("Date")
plt.ylabel("Probability")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()

path = FIGURE_DIR / "sp500_best_deep_learning_probability.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {path}")


## 32. Probability Threshold Analysis

In [ ]:
threshold_rows = []

for name in neural_models:
    probabilities = neural_probabilities[name]

    for threshold in [
        0.50,
        0.55,
        0.60,
        0.65,
        0.70,
    ]:
        threshold_predictions = (
            probabilities >= threshold
        ).astype(int)

        threshold_rows.append({
            "model": name,
            "threshold": threshold,
            "signal_rate": threshold_predictions.mean(),
            "precision": precision_score(
                y_test,
                threshold_predictions,
                zero_division=0,
            ),
            "recall": recall_score(
                y_test,
                threshold_predictions,
                zero_division=0,
            ),
            "f1": f1_score(
                y_test,
                threshold_predictions,
                zero_division=0,
            ),
        })

threshold_df = pd.DataFrame(
    threshold_rows
)

display(threshold_df)

threshold_df.to_csv(
    TABLE_DIR / "sp500_deep_learning_threshold_analysis.csv",
    index=False,
)


## 33. Save Test Predictions

In [ ]:
prediction_output = pd.DataFrame({
    "Date": test_sequence_dates,
    "target": y_test.astype(int),
})

for name in neural_models:
    safe_name = (
        name.lower()
        .replace(" ", "_")
    )

    prediction_output[
        f"{safe_name}_probability_up"
    ] = neural_probabilities[name]

    prediction_output[
        f"{safe_name}_prediction"
    ] = neural_predictions[name]

prediction_path = (
    INTERIM_DIR /
    "sp500_deep_learning_test_predictions.parquet"
)

prediction_output.to_parquet(
    prediction_path,
    index=False,
)

display(prediction_output.head())

print(f"Saved: {prediction_path}")


## 34. Save Training Histories

In [ ]:
history_paths = {}

for name, history in {
    "MLP": mlp_history,
    "LSTM": lstm_history,
    "GRU": gru_history,
}.items():
    safe_name = name.lower()

    path = (
        TABLE_DIR /
        f"sp500_{safe_name}_training_history.csv"
    )

    history.to_csv(
        path,
        index=False,
    )

    history_paths[name] = str(path)

    print(f"{name}: {path}")


## 35. Save Neural-Network Model Artifacts

In [ ]:
model_paths = {}

for name, model in neural_models.items():
    safe_name = name.lower()

    path = (
        MODEL_DIR /
        f"sp500_{safe_name}_classifier.pt"
    )

    torch.save(
        {
            "model_name": name,
            "state_dict": model.state_dict(),
            "sequence_length": SEQUENCE_LENGTH,
            "n_features": len(feature_columns),
            "feature_columns": feature_columns,
            "seed": SEED,
        },
        path,
    )

    model_paths[name] = str(path)

    print(f"{name}: {path}")


## 36. Save Feature-Scaler Artifact

In [ ]:
import joblib

scaler_path = (
    MODEL_DIR /
    "sp500_deep_learning_feature_scaler.pkl"
)

joblib.dump(
    feature_scaler,
    scaler_path,
)

print(f"Saved: {scaler_path}")


## 37. Deep Learning Research Report

In [ ]:
deep_learning_report = {
    "dataset": {
        "rows": int(len(df)),
        "modeling_rows": int(len(model_df)),
        "start": df["Date"].min().strftime("%Y-%m-%d"),
        "end": df["Date"].max().strftime("%Y-%m-%d"),
    },
    "sequence": {
        "sequence_length": SEQUENCE_LENGTH,
        "n_features": len(feature_columns),
        "train_sequences": int(len(X_train)),
        "validation_sequences": int(len(X_val)),
        "test_sequences": int(len(X_test)),
    },
    "training": {
        "batch_size": BATCH_SIZE,
        "max_epochs": EPOCHS,
        "early_stopping_patience": PATIENCE,
        "learning_rate": LEARNING_RATE,
        "device": str(DEVICE),
        "seed": SEED,
    },
    "features": feature_columns,
    "models": neural_metrics_df.to_dict(
        orient="records"
    ),
    "best_model_by_roc_auc": best_dl_model,
    "artifacts": model_paths,
    "scaler": str(scaler_path),
    "methodological_note": (
        "Sequences preserve temporal order. The scaler is fitted on the "
        "training period only. The test period is not used for training "
        "or early stopping. Results are research baselines and require "
        "walk-forward validation before trading interpretation."
    ),
}

report_path = (
    REPORT_DIR /
    "sp500_deep_learning_models_report.json"
)

report_path.write_text(
    json.dumps(
        deep_learning_report,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        deep_learning_report,
        indent=2,
        default=str,
    )
)

print(f"\nSaved: {report_path}")


## 38. Save Sequence Metadata

In [ ]:
sequence_metadata = pd.DataFrame({
    "Date": sequence_dates,
    "target": y_sequences.astype(int),
})

sequence_metadata_path = (
    INTERIM_DIR /
    "sp500_deep_learning_sequence_metadata.parquet"
)

sequence_metadata.to_parquet(
    sequence_metadata_path,
    index=False,
)

print(f"Saved: {sequence_metadata_path}")


## 39. Final Raw Dataset Integrity Check

In [ ]:
master_check = pd.read_csv(
    MASTER_PATH,
    low_memory=False,
)

assert list(master_check.columns) == EXPECTED_COLUMNS
assert len(master_check) == len(df)

master_dates = pd.to_datetime(
    master_check["Date"],
    errors="coerce",
)

assert master_dates.notna().all()
assert master_dates.is_unique
assert master_dates.is_monotonic_increasing

print(
    "Raw master dataset integrity after deep learning: PASS"
)
print(
    f"Master rows: {len(master_check):,}"
)


# Notebook 09 Complete

Notebook 09 has established deep-learning baselines for next-day S&P 500 return direction.

### Models
- MLP
- LSTM
- GRU

### Methodology
- Chronological train/test split
- Training-only feature scaling
- 30-observation temporal sequences
- Chronological validation segment
- Early stopping
- Gradient clipping
- No test-period training

### Outputs
- Accuracy
- Precision
- Recall
- F1
- ROC-AUC
- Classification reports
- Confusion matrices
- Training curves
- Probability threshold analysis
- Test probability predictions
- PyTorch model artifacts
- Feature scaler
- Research report

### Important methodological boundary

These are deep-learning baselines, not final trading models. The next research stages should compare them against classical and tree-based models using leakage-controlled walk-forward validation and trading-aware evaluation.

**Next notebook:** Notebook 10 — Walk-Forward Validation & Model Selection.

Run Notebook 09 from top to bottom and verify the outputs before proceeding.
